# VOLE-Field — durable generative-state reuse

**One proposition, one experiment.** A tiny recurrent video model (ConvLSTM) watches a
synthetic scene and accumulates recurrent state. That state is persisted through
[EntropyFS](https://crates.io/crates/entropyfs) by one process, which then exits. A
*different* process restores the state and generates related futures without replaying the
observed history.

Everything here is native Rust on CPU. **Runtime → Run all.** No GPU, no API key, no
account, no download. The whole notebook is four steps: checkout, build, run, look.

**Notebook revision 3.** Cell 1 prints this revision and checks it against `main`, so
a stale copy announces itself. Colab loads a notebook once and never re-fetches it, so
an old tab can run old cell source against a freshly cloned repository.


In [ ]:
# Cell 1 -- environment and the exact commit under test.
REPO_URL = "https://github.com/infinityabundance/vole-field"
COMMIT = "main"
WORKDIR = "/content/vole-field"

# Revision of *this* notebook. Bump it whenever the notebook changes: cell 1 prints it
# and compares it with the revision on the remote, so a stale copy says so out loud
# rather than failing in a way that looks like a bug in the repository.
NOTEBOOK_REVISION = 3

import json, os, re, shutil, subprocess, urllib.request

def sh(cmd, cwd=None):
    print("\n$ " + " ".join(cmd))
    return subprocess.run(cmd, cwd=cwd, check=True, text=True)

# ---- is this copy the latest? -------------------------------------------------
# This runs before anything that can fail, because a stale notebook's most likely
# symptom is a failure further down -- which is exactly when this needs to have been
# said. Colab loads the .ipynb once and does not re-fetch it on "Run all", so a stale
# tab will happily re-clone the current repository and then execute old cell source.
def notebook_revision(nb):
    """The NOTEBOOK_REVISION a notebook declares, or None if it declares none."""
    text = "\n".join("".join(c.get("source", [])) for c in nb.get("cells", []))
    m = re.search(r"^NOTEBOOK_REVISION\s*=\s*(\d+)\s*$", text, re.M)
    return int(m.group(1)) if m else None


def check_notebook_revision():
    raw = REPO_URL.rstrip("/").replace(
        "https://github.com/", "https://raw.githubusercontent.com/")
    url = "{}/{}/colab/vole_field_demo.ipynb".format(raw, COMMIT)
    print("notebook   : revision {} (this copy)".format(NOTEBOOK_REVISION))
    try:
        req = urllib.request.Request(url, headers={"User-Agent": "vole-field-colab"})
        with urllib.request.urlopen(req, timeout=30) as r:
            remote = notebook_revision(json.load(r))
    except Exception as e:
        print("             could not reach {}: {}".format(url, e))
        return
    if remote is None:
        print("             {} declares no revision".format(COMMIT))
    elif remote == NOTEBOOK_REVISION:
        print("             revision {} on {} -- this copy is current\n"
              .format(remote, COMMIT))
    else:
        print("             revision {} on {} -- THIS COPY IS {}\n"
              .format(remote, COMMIT,
                      "OLD" if remote > NOTEBOOK_REVISION else "NEWER"))
        print("  *** STALE NOTEBOOK: this tab is running an old copy of the cell source. ***")
        print("  Colab does not re-fetch the .ipynb on 'Run all', so re-running keeps using")
        print("  the old code. Close this tab, reopen the notebook, and run again.\n")


check_notebook_revision()

# Rust toolchain. rust-toolchain.toml in the checkout pins the exact compiler, so
# rustup installs that version on first use.
if shutil.which("cargo") is None:
    sh(["bash", "-c",
        "curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs "
        "| sh -s -- -y --profile minimal --default-toolchain none"])
os.environ["PATH"] = os.path.expanduser("~/.cargo/bin") + os.pathsep + os.environ["PATH"]

# Checkout. If you only have a local tree, copy it to WORKDIR first and set REPO_URL
# to a file:// URL, or skip the clone by pre-creating WORKDIR.
if os.path.isdir(os.path.join(WORKDIR, ".git")):
    sh(["git", "-C", WORKDIR, "fetch", "--quiet", "origin"])
elif not os.path.isdir(WORKDIR):
    sh(["git", "clone", "--quiet", REPO_URL, WORKDIR])
sh(["git", "-C", WORKDIR, "checkout", "--quiet", COMMIT])

rev = subprocess.run(["git", "-C", WORKDIR, "rev-parse", "HEAD"],
                     capture_output=True, text=True)
print("\nrepository :", REPO_URL)
if rev.returncode == 0:
    print("commit     :", rev.stdout.strip())
    print(subprocess.run(["git", "-C", WORKDIR, "log", "-1", "--format=%H%n%ad%n%s"],
                         capture_output=True, text=True).stdout)
else:
    print("commit     : (this checkout has no commit yet)")
print("toolchain  :", [l for l in open(os.path.join(WORKDIR, "rust-toolchain.toml"))
                        if l.startswith("channel")][0].strip())
# The pinned compiler comes from rust-toolchain.toml, and rustup only consults
# that file for the *current* directory. rustup was installed above with
# --default-toolchain none, so invoked from anywhere else these fail with
# "no default toolchain configured" and exit 1. cwd=WORKDIR is what selects the
# toolchain -- and this call is also what downloads it on first use.
sh(["rustc", "-V"], cwd=WORKDIR)
sh(["cargo", "-V"], cwd=WORKDIR)

In [ ]:
# Cell 2 -- build. --locked refuses to move any dependency version.
sh(["cargo", "build", "--release", "--locked"], cwd=WORKDIR)

In [ ]:
# Cell 3 -- run. This orchestrates the whole experiment as child processes:
#           producer (earn + persist + exit), baseline (replay + generate),
#           raw checkpoint, and the EntropyFS restore path. The killer table is here.
res = subprocess.run(["cargo", "run", "--release", "--locked"], cwd=WORKDIR,
                     text=True, capture_output=True)
print(res.stdout)
print(res.stderr)
assert res.returncode == 0, f"the run reported FAIL (exit {res.returncode})"

In [ ]:
# Cell 4 -- look at the evidence. Python only draws what Rust produced: no inference,
#           no persistence, no benchmarking, no output comparison.
import csv
from PIL import Image
import matplotlib.pyplot as plt

RUN = os.path.join(WORKDIR, "run")

montage = Image.open(os.path.join(RUN, "branches.ppm"))
fig, ax = plt.subplots(figsize=(10.5, 11.8))
ax.imshow(montage)
ax.axis("off")
ax.set_title(
    "One earned state, four related futures\n"
    "row 1: the observed context tail\n"
    "rows 2-5: the model's continue / turn-left / turn-right / accelerate,\n"
    "          generated after the producing process had already exited\n"
    "rows 6-9: the same four branches, generated by the scene itself (ground truth)",
    fontsize=10,
)
plt.tight_layout()
plt.show()

rows = list(csv.DictReader(open(os.path.join(RUN, "reuse.csv"))))
n = [int(r["N"]) for r in rows]
fig, ax = plt.subplots(figsize=(7.5, 5))
ax.plot(n, [float(r["baseline_ms"]) for r in rows], "o-", label="baseline (replay history)")
ax.plot(n, [float(r["raw_checkpoint_ms"]) for r in rows], "s--", label="raw tensor checkpoint")
ax.plot(n, [float(r["vole_ms"]) for r in rows], "^-", label="VOLE / EntropyFS restore")
ax.set_xscale("log", base=2)
ax.set_xticks(n)
ax.set_xticklabels([str(v) for v in n])
ax.set_xlabel("N related future requests (each in its own fresh process)")
ax.set_ylabel("cumulative wall time (ms)")
ax.set_title("Cumulative cost, including the one-time state persistence")
ax.grid(alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()

print(open(os.path.join(RUN, "reuse.csv")).read())

### Reading the result

* The restored state is **byte-identical** to the state the producer earned, and every
  restored output is **byte-identical** to the from-scratch replay output. That is the
  correctness gate, and it is what makes the timing comparison meaningful.
* The restored path executes **fewer recurrent steps** — that is the load-bearing evidence,
  and it does not depend on the machine being quiet.
* Wall-clock shows whether the saved work was visible on *this* VM. If VOLE is slower than
  the raw checkpoint, or if there is no break-even inside the tested range, the run still
  **passes**: performance is reported, never gatekept.
* The model itself is **not** a good video model, and its control response is sub-pixel.
  That is why the montage shows the scene's own branches beside the model's: you can see
  both what the controls do and what the model makes of them. `cargo run --release -- eval`
  reports the model's quality numerically. Nothing in the claim depends on it.

The full evidence is in `run/results.json`; each child's own report is in `run/reports/`.
`README.md` lists precisely what this experiment does **not** prove.